# Build sum-of-products KEO for water ($\mathrm{H_2O}$) in Jacobi coordinates

In [1]:
import jax
import numpy as np
from jax import numpy as jnp
from scipy import constants

from keosop.gmat import Gmat
from keosop.keo_jax import batch_Gmat, com
from keosop.revad import dot, norm

jax.config.update("jax_enable_x64", True)

G_to_invcm = (
    constants.value("Planck constant")
    * constants.value("Avogadro constant")
    * 1e16
    / (4.0 * np.pi**2 * constants.value("speed of light in vacuum"))
    * 1e5
)

Define the internal-to-Cartesian and Cartesian-to-internal coordinate transformations for $\mathrm{H_2O}$.

The internal coordinates are defined as:

1. $r$ — O-H1 bond length, $\mathbf{r}=\mathbf{r}_{O}-\mathbf{r}_{H_1}$
2. $R$ — distance between O-H1 centre of mass and H2, $\mathbf{R}=\mathbf{r}_{H_2}-(m_{H_1}\mathbf{r}_{H_1}+m_{O}\mathbf{r}_{O})/(m_{H_1}+m_{O})$
3. $\theta$ — angle between the Jacobi vectors, $\theta=\arccos{(\mathbf{r}\cdot\mathbf{R})/(rR)}$

The atom ordering in Cartesian coordinates is:

$$
\mathrm{H_1},\ \mathrm{O},\ \mathrm{H_2}
$$

In [ ]:
masses = np.array([1.0078250, 15.99491463, 1.00782505])  # masses of H, O, H

ncoo = 3


def internal_to_cartesian(q):
    R, r, theta = q

    mA, mB, mC = masses
    mAB = mA + mB

    A = np.array([0.0, 0.0, -mB / mAB * r], dtype=object)
    B = np.array([0.0, 0.0, mA / mAB * r], dtype=object)
    C = np.array([R * np.sin(theta), 0.0, R * np.cos(theta)], dtype=object)

    return np.array([A, B, C], dtype=object)


def cartesian_to_internal(xyz):
    A, B, C = xyz

    mA, mB, mC = masses
    mAB = mA + mB

    rvec = B - A
    r = norm(rvec)

    comAB = (mA * A + mB * B) / mAB
    Rvec = C - comAB
    R = norm(Rvec)

    theta = np.arccos(dot(rvec, Rvec) / (r * R))

    return np.array([R, r, theta], dtype=object)

Define one-dimensional grids for the internal coordinate representation and construct the G-matrix in sum-of-products form

In [3]:
grids = [
    np.linspace(0.8, 1.2, 5),  # r1
    np.linspace(0.8, 1.2, 5),  # r2
    np.linspace(60 * np.pi / 180, 170 * np.pi / 180, 5),  # alpha
]

gmat = Gmat(grids, masses, internal_to_cartesian, cartesian_to_internal)
gvib_sop, grovib_sop, grot_sop = gmat.run(return_rot=True)

print("Number of terms in Gvib_{i,j}\n i  j no.terms")
for i in range(ncoo):
    for j in range(i, ncoo):
        print(f"{i:2d} {j:2d} {gvib_sop[i, j].nterms:6d}")

print("\nNumber of terms in Grovib_{i,j}\n i  j no.terms")
for i in range(3):
    for j in range(ncoo):
        print(f"{i:2d} {j:2d} {grovib_sop[i, j].nterms:6d}")

print("\nNumber of terms in Grot_{i,j}\n i  j no.terms")
for i in range(3):
    for j in range(i, 3):
        print(f"{i:2d} {j:2d} {grot_sop[i, j].nterms:6d}")

Number of terms in Gvib_{i,j}
 i  j no.terms
 0  0      1
 0  1      1
 0  2      1
 1  1      1
 1  2      2
 2  2      5

Number of terms in Grovib_{i,j}
 i  j no.terms
 0  0      1
 0  1      1
 0  2      1
 1  0      1
 1  1      1
 1  2      1
 2  0      1
 2  1      1
 2  2      1

Number of terms in Grot_{i,j}
 i  j no.terms
 0  0      1
 0  1      1
 0  2      1
 1  1      1
 1  2      1
 2  2      2


Evaluate the G-matrix numerically on the multidimensional grid formed by the Cartesian product of the one-dimensional grids used to construct the sum-of-products representation

In [5]:
@com(masses)
def internal_to_cartesian_jax(q):
    R, r, theta = q

    mA, mB, mC = masses
    mAB = mA + mB

    A = jnp.array([0.0, 0.0, -mB / mAB * r], dtype=jnp.float64)
    B = jnp.array([0.0, 0.0, mA / mAB * r], dtype=jnp.float64)
    C = jnp.array([R * jnp.sin(theta), 0.0, R * jnp.cos(theta)], dtype=jnp.float64)

    return jnp.array([A, B, C], dtype=jnp.float64)


grid_ndim = np.stack(np.meshgrid(*grids, indexing="ij"), axis=-1)
grid_ndim = grid_ndim.reshape(-1, len(grids))

gmat = batch_Gmat(grid_ndim, masses, internal_to_cartesian_jax)

gvib_ndim = gmat[:, :ncoo, :ncoo]
grovib_ndim = gmat[:, ncoo : ncoo + 3, :ncoo]
grot_ndim = gmat[:, ncoo : ncoo + 3, ncoo : ncoo + 3]

Calculate point-wise differences between the sum-of-products and numerical G-matrix

In [6]:
# number of multidimensional grid points
ngrid = grid_ndim.shape[0]

# evaluate the sum-of-products G-matrix on the full direct-product grid
gvib_sop_eval = np.zeros((ngrid, ncoo, ncoo), dtype=np.float64)
grot_sop_eval = np.zeros((ngrid, 3, 3), dtype=np.float64)
grovib_sop_eval = np.zeros((ngrid, 3, ncoo), dtype=np.float64)

# map each multidimensional grid point to its 1D grid indices
grid_indices = np.indices([len(grid) for grid in grids]).reshape(len(grids), -1).T

# Gvib
for ipoint, inds in enumerate(grid_indices):
    for i in range(ncoo):
        for j in range(ncoo):
            sop = gvib_sop[i, j]

            value = 0.0

            for iterm in range(sop.nterms):
                term = 1.0

                for k in range(ncoo):
                    term *= sop.terms[iterm][k][inds[k]]

                value += term

            gvib_sop_eval[ipoint, i, j] = value * G_to_invcm

# Grot
for ipoint, inds in enumerate(grid_indices):
    for i in range(3):
        for j in range(3):
            sop = grot_sop[i, j]

            value = 0.0

            for iterm in range(sop.nterms):
                term = 1.0

                for k in range(ncoo):
                    term *= sop.terms[iterm][k][inds[k]]

                value += term

            grot_sop_eval[ipoint, i, j] = value * G_to_invcm

# Grovib
for ipoint, inds in enumerate(grid_indices):
    for i in range(3):
        for j in range(ncoo):
            sop = grovib_sop[i, j]

            value = 0.0

            for iterm in range(sop.nterms):
                term = 1.0

                for k in range(ncoo):
                    term *= sop.terms[iterm][k][inds[k]]

                value += term

            grovib_sop_eval[ipoint, i, j] = value * G_to_invcm

In [7]:
# differences

diff_vib = gvib_sop_eval - gvib_ndim
diff_rovib = grovib_sop_eval - grovib_ndim
diff_rot = grot_sop_eval - grot_ndim

abs_diff_vib = np.abs(diff_vib)
abs_diff_rovib = np.abs(diff_rovib)
abs_diff_rot = np.abs(diff_rot)

print("--- Gvib ---")
print("Maximum absolute difference:", np.max(abs_diff_vib))
print("Mean absolute difference:", np.mean(abs_diff_vib))
print("RMS difference:", np.sqrt(np.mean(diff_vib**2)))

print("\n--- Grovib ---")
print("Maximum absolute difference:", np.max(abs_diff_rovib))
print("Mean absolute difference:", np.mean(abs_diff_rovib))
print("RMS difference:", np.sqrt(np.mean(diff_rovib**2)))

print("\n--- Grot ---")
print("Maximum absolute difference:", np.max(abs_diff_rot))
print("Mean absolute difference:", np.mean(abs_diff_rot))
print("RMS difference:", np.sqrt(np.mean(diff_rot**2)))

--- Gvib ---
Maximum absolute difference: 5.968558980384842e-13
Mean absolute difference: 1.8503284409155086e-14
RMS difference: 5.540317140259386e-14

--- Grovib ---
Maximum absolute difference: 6.394884621840902e-14
Mean absolute difference: 1.4125920586758904e-15
RMS difference: 5.524527938626721e-15

--- Grot ---
Maximum absolute difference: 2.7284841053187847e-12
Mean absolute difference: 2.6509954725245026e-14
RMS difference: 1.6687482670441293e-13


In [8]:
print("--- Gvib element-wise errors ---")

for i in range(ncoo):
    for j in range(i, ncoo):
        d = diff_vib[:, i, j]
        max_abs = np.max(np.abs(d))
        rms = np.sqrt(np.mean(d**2))

        print(
            f"Gvib[{i},{j}]  "
            f"max abs = {max_abs:.3e}   "
            f"RMS = {rms:.3e}   "
        )

print("\n--- Grovib element-wise errors ---")

for i in range(3):
    for j in range(ncoo):
        d = diff_rovib[:, i, j]
        max_abs = np.max(np.abs(d))
        rms = np.sqrt(np.mean(d**2))

        print(
            f"Grovib[{i},{j}]  "
            f"max abs = {max_abs:.3e}   "
            f"RMS = {rms:.3e}   "
        )

print("\n--- Grot element-wise errors ---")

for i in range(3):
    for j in range(i, 3):
        d = diff_rot[:, i, j]
        max_abs = np.max(np.abs(d))
        rms = np.sqrt(np.mean(d**2))

        print(
            f"Grot[{i},{j}]  "
            f"max abs = {max_abs:.3e}   "
            f"RMS = {rms:.3e}   "
        )

--- Gvib element-wise errors ---
Gvib[0,0]  max abs = 1.421e-14   RMS = 1.101e-14   
Gvib[0,1]  max abs = 1.949e-16   RMS = 1.191e-16   
Gvib[0,2]  max abs = 7.900e-15   RMS = 2.357e-15   
Gvib[1,1]  max abs = 7.105e-15   RMS = 7.105e-15   
Gvib[1,2]  max abs = 1.498e-13   RMS = 6.418e-14   
Gvib[2,2]  max abs = 5.969e-13   RMS = 1.386e-13   

--- Grovib element-wise errors ---
Grovib[0,0]  max abs = 0.000e+00   RMS = 0.000e+00   
Grovib[0,1]  max abs = 0.000e+00   RMS = 0.000e+00   
Grovib[0,2]  max abs = 0.000e+00   RMS = 0.000e+00   
Grovib[1,0]  max abs = 3.411e-15   RMS = 1.284e-15   
Grovib[1,1]  max abs = 2.105e-17   RMS = 5.532e-18   
Grovib[1,2]  max abs = 6.395e-14   RMS = 1.652e-14   
Grovib[2,0]  max abs = 0.000e+00   RMS = 0.000e+00   
Grovib[2,1]  max abs = 0.000e+00   RMS = 0.000e+00   
Grovib[2,2]  max abs = 0.000e+00   RMS = 0.000e+00   

--- Grot element-wise errors ---
Grot[0,0]  max abs = 3.553e-14   RMS = 1.192e-14   
Grot[0,1]  max abs = 0.000e+00   RMS = 0.000e+0

In [9]:
print("--- Gvib largest discrepancy ---")
idx = np.unravel_index(np.argmax(abs_diff_vib), abs_diff_vib.shape)
ipoint, i, j = idx

print("grid point :", ipoint)
print("coordinates:", np.asarray(grid_ndim[ipoint]))
print("element    :", (i, j))
print("SOP        :", gvib_sop_eval[ipoint, i, j])
print("reference  :", gvib_ndim[ipoint, i, j])
print("difference :", diff_vib[ipoint, i, j])

print("\n--- Grovib largest discrepancy ---")
idx = np.unravel_index(np.argmax(abs_diff_rovib), abs_diff_rovib.shape)
ipoint, i, j = idx

print("grid point :", ipoint)
print("coordinates:", np.asarray(grid_ndim[ipoint]))
print("element    :", (i, j))
print("SOP        :", grovib_sop_eval[ipoint, i, j])
print("reference  :", grovib_ndim[ipoint, i, j])
print("difference :", diff_rovib[ipoint, i, j])

print("\n--- Grot largest discrepancy ---")
idx = np.unravel_index(np.argmax(abs_diff_rot), abs_diff_rot.shape)
ipoint, i, j = idx

print("grid point :", ipoint)
print("coordinates:", np.asarray(grid_ndim[ipoint]))
print("element    :", (i, j))
print("SOP        :", grot_sop_eval[ipoint, i, j])
print("reference  :", grot_ndim[ipoint, i, j])
print("difference :", diff_rot[ipoint, i, j])

--- Gvib largest discrepancy ---
grid point : 24
coordinates: [0.8        1.2        2.96705973]
element    : (np.int64(2), np.int64(2))
SOP        : 80.06478489127498
reference  : 80.06478489127439
difference : 5.968558980384842e-13

--- Grovib largest discrepancy ---
grid point : 78
coordinates: [1.1        0.8        2.48709418]
element    : (np.int64(1), np.int64(2))
SOP        : -55.5646225909511
reference  : -55.56462259095104
difference : -6.394884621840902e-14

--- Grot largest discrepancy ---
grid point : 4
coordinates: [0.8        0.8        2.96705973]
element    : (np.int64(2), np.int64(2))
SOP        : 3623.3888026676254
reference  : 3623.3888026676227
difference : 2.7284841053187847e-12
